In [1]:
# Importing necessary libraries
import os
import sys
from pathlib import Path
import argparse
import logging
import time
import json
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on the Python path so scripts/enhancement imports work
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))


In [ ]:
# import  dataset 

import pandas as pd, json, pathlib

mnl_path = r"\\users\\users\\hisham\\EUROMOD-STORAGE\\Data\\processed\\fr\\2016\\fr_2016_RURO_mnl.parquet"
df = pd.read_parquet(mnl_path)  # uses pyarrow/fastparquet if available

meta_path = pathlib.Path(mnl_path).with_suffix('.meta.json')
if meta_path.exists():
    meta = json.loads(meta_path.read_text())



In [3]:
df.info() # Display dataframe information
print(df.head()) # Display first few rows of the dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448900 entries, 0 to 448899
Columns: 1501 entries, idhh to year_16
dtypes: bool(1), float64(1470), int64(23), object(7)
memory usage: 5.0+ GB
      idhh     idperson  idmother  idfather  idpartner   idorighh  \
0  1495800  149580001.0       0.0       0.0        0.0  1495800.0   
1  1495800  149580001.0       0.0       0.0        0.0  1495800.0   
2  1495800  149580001.0       0.0       0.0        0.0  1495800.0   
3  1495800  149580001.0       0.0       0.0        0.0  1495800.0   
4  1495800  149580001.0       0.0       0.0        0.0  1495800.0   

   idorigperson   dag  dgn  dec  ...  drgn1_0  drgn1_2  drgn1_3  drgn1_4  \
0   149580001.0  51.0  0.0  0.0  ...        0        0        0        0   
1   149580001.0  51.0  0.0  0.0  ...        0        0        0        0   
2   149580001.0  51.0  0.0  0.0  ...        0        0        0        0   
3   149580001.0  51.0  0.0  0.0  ...        0        0        0        0   
4   149580001

In [8]:
import pandas as pd, json, pathlib

orig_path = r"\\users\\users\\hisham\\EUROMOD-STORAGE\\Data\\processed\\fr\\2016\\fr_2016_singles_male.parquet"
df_orig = pd.read_parquet(orig_path)  # uses pyarrow/fastparquet if available

df_orig.info() # Display dataframe information
print(df_orig.head()) # Display first few rows of the dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Columns: 408 entries, idhh to num_children_11_17
dtypes: bool(2), float64(388), int64(17), object(1)
memory usage: 2.8+ MB
        idhh     idperson  idmother  idfather  idpartner   idorighh  \
0  1504300.0  150430001.0       0.0       0.0        0.0  1504300.0   
1  1526601.0  152660101.0       0.0       0.0        0.0  1526601.0   
2  1527000.0  152700001.0       0.0       0.0        0.0  1527000.0   
3  1531200.0  153120001.0       0.0       0.0        0.0  1531200.0   
4  1533500.0  153350001.0       0.0       0.0        0.0  1533500.0   

   idorigperson   dag  dgn  dec  ...  diff_yem_final  flag_periodicity  \
0   150430001.0  38.0  1.0  0.0  ...             0.0              True   
1   152660003.0  27.0  1.0  0.0  ...             0.0              True   
2   152700001.0  28.0  1.0  0.0  ...             0.0              True   
3   153120001.0  63.0  1.0  0.0  ...             NaN             False   
4   1533

In [ ]:
from biogeme import biogeme
from biogeme import models
from biogeme.dfexpressions import Beta, log
from biogeme.database import Database



u:\Desktop\Nizam_Hisham\MNL\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [6]:

df_male_s = df.query("ruro_group == 1 and dgn == 1")  # singles, male

# 2) Inspect the structure to map alternative/choice columns
print(df_male_s.head())
print(df_male_s.columns)


        idhh     idperson  idmother  idfather  idpartner   idorighh  \
400  1504300  150430001.0       0.0       0.0        0.0  1504300.0   
401  1504300  150430001.0       0.0       0.0        0.0  1504300.0   
402  1504300  150430001.0       0.0       0.0        0.0  1504300.0   
403  1504300  150430001.0       0.0       0.0        0.0  1504300.0   
404  1504300  150430001.0       0.0       0.0        0.0  1504300.0   

     idorigperson   dag  dgn  dec  ...  drgn1_0  drgn1_2  drgn1_3  drgn1_4  \
400   150430001.0  38.0  1.0  0.0  ...        0        0        0        0   
401   150430001.0  38.0  1.0  0.0  ...        0        0        0        0   
402   150430001.0  38.0  1.0  0.0  ...        0        0        0        0   
403   150430001.0  38.0  1.0  0.0  ...        0        0        0        0   
404   150430001.0  38.0  1.0  0.0  ...        0        0        0        0   

     drgn1_5  drgn1_6  drgn1_7  drgn1_8  year  year_16  
400        0        0        0        0  16.0  

In [ ]:


def grad_negative_log_likelihood(theta, data, structure):
    scores = score_matrix(theta, data, structure)  # shape (N, K)
    return -scores.sum(axis=0)

def estimate(
    gender_key: str,
    df: pd.DataFrame,
    labels: Tuple[str, ...],
    *,
    include_ascs: bool,
    gender_column: str,
    output_dir: Path,
    log_level: int,
    c_scale_quantile: float,
    variant: str,
    gender_split: bool = False,
    z_by_gender: bool = False,
    model_prefix: str | None = None,
    analyzer_source: str = "mle",
    dataset_source_dir: Path | None = None,
    analyzer_base_dir: Path | None = None,
) -> None:
    if gender_key != "pooled":
        gender_split = False
        z_by_gender = False
    data = prepare_dataset(df, labels, gender_column=gender_column, c_scale_quantile=c_scale_quantile)
    if gender_split and not data.has_gender_param:
        gender_split = False
        z_by_gender = False
    structure = build_param_structure(
        labels,
        data,
        include_ascs=include_ascs,
        pooled=(gender_key == "pooled"),
        gender_split=gender_split,
        z_by_gender=z_by_gender,
    )

    # Compute medians/modes for Z shifters (mode for binary, median otherwise)
    def _is_binary01(arr: np.ndarray) -> bool:
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            return False
        u = np.unique(finite)
        return len(u) <= 2 and set(np.round(u, 6)).issubset({0.0, 1.0})

    Z_stats: Dict[str, float] = {}
    _zmap = {
        "age_norm": "age",
        "age2_norm": "age2",
        "child_norm": "child",
        "dch": "dch",
        "gender": "gender",
    }
    for fk, arr in data.features.items():
        name = _zmap.get(fk, fk)
        a = np.asarray(arr, dtype=float)
        if _is_binary01(a):
            finite = a[np.isfinite(a)]
            if finite.size:
                vals, counts = np.unique(finite, return_counts=True)
                val = float(vals[np.argmax(counts)])
            else:
                val = 0.0
        else:
            val = float(np.nanmedian(a))
        Z_stats[name] = val

    # medians of normalized consumption & leisure at actual choices
    idx = np.arange(len(data.actual_idx))
    C_norm_actual = data.C_norm[idx, data.actual_idx]
    L_norm_actual = data.L_norm[idx, data.actual_idx]
    c_norm_med = float(np.nanmedian(C_norm_actual))
    l_norm_med = float(np.nanmedian(L_norm_actual))

    LOGGER.info("[%s] Parameter vector: %s", gender_key, structure.param_names)

    theta0 = initial_theta(structure)
    bounds: List[Tuple[float | None, float | None]] = []
    for name in structure.param_names:
        bounds.append((None, None)) # No box constraints

    t_start = time.perf_counter()
   
    t_start = time.perf_counter()
    result = minimize(
        negative_log_likelihood,
        theta0,
        args=(data, structure),
        method="L-BFGS-B",
        jac = grad_negative_log_likelihood,
        bounds=bounds,
        options={"maxiter": 2000, "disp": log_level <= logging.DEBUG},
    )
    solve_time = time.perf_counter() - t_start
    solve_time = time.perf_counter() - t_start

    if not result.success:
        LOGGER.warning("[%s] Optimiser did not converge: %s", gender_key, result.message)

    theta_hat = result.x
    ll_star = -negative_log_likelihood(theta_hat, data, structure)
    ll_null = compute_null_loglik(data)

    param_values = flatten_params(theta_hat, structure)

    # Leisure slope at median Z (scalar)
    beta_l_med = float(param_values.get("beta_l0", 0.0))
    for dname in structure.delta_names:
        base = dname.replace("delta_", "")
        zval = Z_stats.get(base, 0.0)
        beta_l_med += float(param_values.get(dname, 0.0)) * float(zval)

    alpha_c = float(param_values.get("alpha_c", 0.0))
    alpha_l = float(param_values.get("alpha_l", 0.0))
    beta_c = float(param_values.get("beta_c", 0.0))

    # Normalized marginal utilities (NO division by y_ref or T)
    # MUC^norm(c_norm) = beta_c * c_norm^(alpha_c - 1)
    # MUL^norm(l_norm) = beta_l_med * l_norm^(alpha_l - 1)
    muc_norm_med = beta_c * (c_norm_med ** (alpha_c - 1.0)) if c_norm_med > 0 else float("nan")
    mul_norm_med = beta_l_med * (l_norm_med ** (alpha_l - 1.0)) if l_norm_med > 0 else float("nan")

    # Normalized MRS at median (dimensionless): MUL^norm / MUC^norm
    mrs_norm_med = (
        mul_norm_med / muc_norm_med
        if (np.isfinite(mul_norm_med) and np.isfinite(muc_norm_med) and muc_norm_med != 0.0)
        else float("nan")
    )

    # Zero-crossings in (0,1] only occur if slope coef is exactly zero
    muc_norm_zero_c = None if beta_c != 0.0 else 0.0
    mul_norm_zero_l = None if beta_l_med != 0.0 else 0.0

    predicted = predict_choices(theta_hat, data, structure)
    accuracy = float(np.mean(predicted == data.actual_idx))

    k_params = len(structure.param_names)
    n_obs = len(data.actual_idx)
    rho2 = float(1.0 - ll_star / ll_null) if ll_null != 0 else float("nan")
    rho2_adj = float(1.0 - (ll_star - k_params) / ll_null) if ll_null != 0 else float("nan")
    aic = 2 * k_params - 2 * ll_star
    bic = math.log(n_obs) * k_params - 2 * ll_star

    cm = confusion_matrix(data.actual_choice, predicted, labels)

    scores = score_matrix(theta_hat, data, structure)

    # First-order condition diagnostic (sum of per-obs scores should be ~0 at optimum)
    foc_norm = float(np.linalg.norm(scores.sum(axis=0)))
    LOGGER.info("[%s] FOC check: ||Σ_i s_i(θ̂)|| = %.3e", gender_key, foc_norm)

    # Recompute observed Hessian of the *sum* NLL by symmetric central differences (scale-aware)
    H = approximate_hessian(theta_hat, data, structure, eps=None)
    H = 0.5 * (H + H.T)

    # Light Tikhonov ridge for numerical stability (scale-aware)
    ridge = 1e-8 * max(1.0, float(np.mean(np.abs(np.diag(H)))))
    H = H + ridge * np.eye(k_params)

    # Invert observed information
    Hinv = np.linalg.inv(H)

    # Classical covariance = inverse observed information
    cov = Hinv.copy()

    # Robust (sandwich) covariance: H^{-1} (sum_i s_i s_i^T) H^{-1}
    G = scores.T @ scores  # sum over observations
    cov_rob = Hinv @ G @ Hinv
    cov_rob = 0.5 * (cov_rob + cov_rob.T)

    # Diagnostics on curvature (optional but handy)
    w, _ = np.linalg.eigh(0.5 * (H + H.T))
    min_eig_H = float(np.min(w))
    max_eig_H = float(np.max(w))
    cond_H = float(max_eig_H / max(min_eig_H, 1e-16))
    LOGGER.info("[%s] Observed-Hessian eigs: min=%.3e  max=%.3e  cond≈%.3e", gender_key, min_eig_H, max_eig_H, cond_H)

    values_vector = np.array([param_values[name] for name in structure.param_names], dtype=float)

    diag_cov = np.diag(cov)
    diag_cov_rob = np.diag(cov_rob)
    se = np.sqrt(np.where(diag_cov >= 0, diag_cov, np.nan))
    se_rob = np.sqrt(np.where(diag_cov_rob >= 0, diag_cov_rob, np.nan))

    with np.errstate(divide="ignore", invalid="ignore"):
        t_values = np.divide(values_vector, se, out=np.full_like(values_vector, np.nan), where=se > 0)
        t_values_rob = np.divide(values_vector, se_rob, out=np.full_like(values_vector, np.nan), where=se_rob > 0)

    p_values = np.where(np.isnan(t_values), np.nan, 2.0 * norm.sf(np.abs(t_values)))
    p_values_rob = np.where(np.isnan(t_values_rob), np.nan, 2.0 * norm.sf(np.abs(t_values_rob)))

    param_df = build_parameter_dataframe(
        param_values,
        structure,
        se=se,
        t_values=t_values,
        p_values=p_values,
        se_rob=se_rob,
        t_values_rob=t_values_rob,
        p_values_rob=p_values_rob,
    )

    draws_df, muc_summary = generate_mucmul_draws(param_values, data)

    min_eig = float(np.min(np.linalg.eigvalsh(0.5 * (cov + cov.T))))

    LOGGER.info("[%s] Log-likelihood at optimum: %.4f (solve time %.3fs)", gender_key, ll_star, solve_time)
    LOGGER.info("[%s] LL(null)=%.4f  rho²=%.4f  rho²_adj=%.4f", gender_key, ll_null, rho2, rho2_adj)
    LOGGER.info("[%s] AIC=%.2f  BIC=%.2f", gender_key, aic, bic)
    LOGGER.info("[%s] Accuracy=%.2f%%", gender_key, accuracy * 100.0)
    LOGGER.info("[%s] Confusion matrix:\n%s", gender_key, cm)
    LOGGER.info("[%s] Parameter count K=%d  min_eig(cov)=%.3e", gender_key, k_params, min_eig)
    LOGGER.info(
        "[%s] Share MUC<0 actual=%.2f%%  MUL<0 actual=%.2f%%",
        gender_key,
        muc_summary["share_MUC_actual_neg"] * 100.0,
        muc_summary["share_MUL_actual_neg"] * 100.0,
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    base_prefix = model_prefix or f"boxcox_{gender_key}"
    model_name = f"{base_prefix}_{variant}".replace(".", "_")
    meta = {
        "spec": "boxcox",
        "labels": list(labels),
        "include_ascs": include_ascs,
        "pooled": gender_key == "pooled",
        "variant": variant,
        "c_scale_quantile": c_scale_quantile,
        "log_likelihood": ll_star,
        "null_log_likelihood": ll_null,
        "rho2": rho2,
        "rho2_adj": rho2_adj,
        "aic": aic,
        "bic": bic,
        "accuracy": accuracy,
        "k_params": k_params,
        "min_eig_cov": min_eig,
        "y_ref": data.y_ref,
        "T": T_HOURS,
        "parameters": param_values,
        "n_obs": n_obs,
    }
    if dataset_source_dir is not None:
        meta["data_dir"] = str(dataset_source_dir)
    meta.update({k: float(v) for k, v in muc_summary.items()})
    meta.update({
        "parameters_csv": f"{model_name}_parameters.csv",
        "confusion_csv": f"{model_name}_confusion.csv",
        "Z_medians_or_modes": Z_stats,
        "c_norm_median": c_norm_med,
        "l_norm_median": l_norm_med,
        "MUC_norm_med": float(muc_norm_med),
        "MUL_norm_med": float(mul_norm_med),
        "MRS_norm_med": float(mrs_norm_med),
        "muc_norm_zero_c_norm": muc_norm_zero_c,
        "mul_norm_zero_l_norm": mul_norm_zero_l,
    })

    # --- Standardized run metadata for analyzers ---
    # Dataset summary
    n_obs_meta = meta.get("n_obs")
    try:
        if n_obs_meta is None and "n_obs" in locals():
            n_obs_meta = int(n_obs)
    except Exception:
        pass

    # Try to infer years in sample if the wide dataset has a 'year' column
    years = meta.get("years")
    if years is None:
        try:
            df_for_years = df if "df" in locals() else None
            if df_for_years is not None and "year" in df_for_years.columns:
                years = sorted(
                    int(y) for y in df_for_years["year"].dropna().unique().tolist()
                )
        except Exception:
            years = None

    # Gender tag is already known in this loop
    gender_tag = meta.get("gender", None)
    if gender_tag is None and "gender_key" in locals():
        gender_tag = gender_key

    # Timestamp of estimation (local time)
    run_timestamp = datetime.datetime.now().isoformat(timespec="seconds")

    # Attach run summary to meta
    meta.update(
        {
            "run_timestamp": run_timestamp,
            "solve_time_sec": float(solve_time) if "solve_time" in locals() else None,
            "ll_star": float(ll_star) if "ll_star" in locals() else None,
            "ll_null": float(ll_null) if "ll_null" in locals() else None,
            "rho2": float(rho2) if "rho2" in locals() else None,
            "rho2_adj": float(rho2_adj) if "rho2_adj" in locals() else None,
            "aic": float(aic) if "aic" in locals() else None,
            "bic": float(bic) if "bic" in locals() else None,
            "score_norm": float(foc_norm) if "foc_norm" in locals() else None,
            "hess_min_eig": float(min_eig_H) if "min_eig_H" in locals() else None,
            "hess_max_eig": float(max_eig_H) if "max_eig_H" in locals() else None,
            "hess_cond": float(cond_H) if "cond_H" in locals() else None,
            "accuracy": float(accuracy) if "accuracy" in locals() else None,
            "n_obs": int(n_obs_meta) if n_obs_meta is not None else None,
            "years": years,
            "gender": gender_tag,
        }
    )

    write_parameter_metadata(output_dir, model_name, param_df, meta)

    cm_path = output_dir / f"{model_name}_confusion.csv"
    cm.to_csv(cm_path)

    draws_path = output_dir / f"{model_name}_mucmul_draws.csv"
    draws_df.to_csv(draws_path)

    muc_summary_path = output_dir / f"{model_name}_mucmul_summary.json"
    muc_summary_path.write_text(json.dumps(muc_summary, indent=2), encoding="utf-8")

    run_analyzer(analyzer_source, [gender_key], variant, data_dir=dataset_source_dir, base_dir=analyzer_base_dir)
